<a href="https://colab.research.google.com/github/Harsh-Prajapati54/LLMs---Playbook/blob/main/Text_Embedding_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Creating a Text Embedding models

#### Refrence for this notebook is Hands-on-Large Language models

### Creating a Contrastive Learning model

In [1]:
from datasets import load_dataset

# load MNLI dataset from GLUE
# this dataset is based on ( entailment , contradiction , neutral)

train_dataset = load_dataset("nyu-mll/glue", "mnli", split="train").select(range(100000))

train_dataset = train_dataset.remove_columns("idx")

lets look at dataset !!!

In [2]:
"""
    lets look at the structure of an datasets
    it contains 100000 rows having features: ['premise', 'hypothesis', 'label']

    here the term label have an 3 numbers from ( 0, 1, 2)
    0 = entailment  (both hypothesis and premises will be similar)
    1 = neutral     (both hypothesis and premises will be neutral)
    2 = contradiction (both hypothesis and premises will be opposite)
"""

train_dataset

Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 100000
})

In [3]:
train_dataset[2]

{'premise': 'One of our number will carry out your instructions minutely.',
 'hypothesis': 'A member of my team will execute your orders with immense precision.',
 'label': 0}

### Train an Embedding modal from scratch

In [4]:
from sentence_transformers import SentenceTransformer

# use a base modal
embedding_modal = SentenceTransformer("bert-base-uncased")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


now we need an loss function for sentence transformer , we will use `softmax` function for it .

In [5]:
from sentence_transformers import  losses

# define the loss function. in softmax loss, we will also needd to  set the number of labels

train_loss = losses.SoftmaxLoss(
    model = embedding_modal,
    sentence_embedding_dimension=embedding_modal.get_sentence_embedding_dimension(),
    num_labels=3
)

/tmp/ipykernel_11567/2883587312.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import  losses
/tmp/ipykernel_11567/2883587312.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  sentence_embedding_dimension=embedding_modal.get_sentence_embedding_dimension(),


Before training our model we need to create an evaluation metric to evaluate our modal ,

lets use Sementic Textual Similiraty Benchmark(STSB) , it is acollection of an human labeled dataset from sementic ranking form 1 to 5

we use this dataset to explore how our modal scores on this semantic simealrity tasks

In [6]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

val_sts = load_dataset("nyu-mll/glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1 = val_sts["sentence1"],
    sentences2 = val_sts["sentence2"],
    scores = [score/5 for score in val_sts["label"]],
    main_similarity = "cosine"
)

/tmp/ipykernel_11567/2941289755.py:1: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator


In [7]:
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir = "base_embedding_model",
    num_train_epochs = 1,
    per_device_train_batch_size = 64,
    per_device_eval_batch_size = 64,
    warmup_steps = 100,
    fp16 = True,
    logging_steps = 100,
    eval_steps = 100
)



/tmp/ipykernel_11567/812423036.py:1: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import SentenceTransformerTrainingArguments


Till now we have set our `dataset` , `embedding modal` `loss` and `evaluator`, now we can start training the modal using `SentenceTransformerTrainer`

In [8]:
from sentence_transformers.trainer import  SentenceTransformerTrainer

trainer = SentenceTransformerTrainer(
    model = embedding_modal,
    args = args,
    train_dataset = train_dataset,
    loss = train_loss,
    evaluator = evaluator
)

trainer.train()


/tmp/ipykernel_11567/2585222918.py:1: DeprecationWarning: Importing from 'sentence_transformers.trainer' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.trainer' instead.
  from sentence_transformers.trainer import  SentenceTransformerTrainer


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

dataset = dataset.select_columns(['hypothesis', 'entailment', 'contradiction'])


Step,Training Loss
100,1.059463
200,0.893074
300,0.853840
400,0.814316
500,0.794432
600,0.769217
700,0.746059
800,0.755298
900,0.734979
1000,0.728191


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1563, training_loss=0.7769589414980956, metrics={'train_runtime': 571.0076, 'train_samples_per_second': 175.129, 'train_steps_per_second': 2.737, 'total_flos': 0.0, 'train_loss': 0.7769589414980956, 'epoch': 1.0})

In [9]:
evaluator(embedding_modal)

{'pearson_cosine': 0.6198747893981736, 'spearman_cosine': 0.6768087215213209}

## Cosine Similarity as loss function

In Cosine Similarity loss function it minimize the distance of similar sentences and maximize the distance of disimilar sentences , using similarity score produced by

osine similarity is a metric used to measure how similar two vectors are, irrespective of their magnitude. It measures the cosine of the angle between two non-zero vectors projected in a multi-dimensional space.

## Vector Notation

$$\text{Cosine Similarity} = \cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$$

it gives an answer between 0 to 1 so we need to convert the entailment neutral and contradiction to 1 to 0

neutral and contradiction is dissimilar so == 0
and entailment == 1   

In [10]:
# (neutral / contradiction )  = 0 , entailment = 1
from datasets import Dataset,load_dataset
mapping = {2:0,1:0,0:1}

train_dataset = Dataset.from_dict({
    "sentence1": train_dataset["premise"],
    "sentence2": train_dataset["hypothesis"],
    "label": [mapping[label] for label in train_dataset["label"]]
})


### Evaluator

same as above

In [11]:
evaluator = EmbeddingSimilarityEvaluator(
    sentences1 = val_sts["sentence1"],
    sentences2 = val_sts["sentence2"],
    scores = [score/5 for score in val_sts["label"]],
    main_similarity = "cosine"
)

### loss function

In [12]:
train_loss = losses.CosineSimilarityLoss(model = embedding_modal)

In [13]:
# training arguments
args = SentenceTransformerTrainingArguments(
    output_dir = "cosineloss_embedding_model",
    num_train_epochs = 1,
    per_device_train_batch_size = 64,
    per_device_eval_batch_size = 64,
    warmup_steps = 100,
    fp16 = True,
    logging_steps = 100,
    eval_steps = 100
)


### Train the modal  

In [14]:
trainer = SentenceTransformerTrainer(
    model = embedding_modal,
    args = args,
    train_dataset = train_dataset,
    loss = train_loss,
    evaluator = evaluator
)

trainer.train()

Step,Training Loss
100,0.169889
200,0.128183
300,0.114683
400,0.102449
500,0.096046
600,0.098801
700,0.096513
800,0.099880
900,0.102914
1000,0.106103


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1563, training_loss=0.11554871837984501, metrics={'train_runtime': 617.9669, 'train_samples_per_second': 161.821, 'train_steps_per_second': 2.529, 'total_flos': 0.0, 'train_loss': 0.11554871837984501, 'epoch': 1.0})

In [15]:
evaluator = EmbeddingSimilarityEvaluator(
    sentences1 = val_sts["sentence1"],
    sentences2 = val_sts["sentence2"],
    scores = [score/5 for score in val_sts["label"]],
    main_similarity = "cosine"
)
evaluator(embedding_modal)

{'pearson_cosine': 0.7107027943065035, 'spearman_cosine': 0.7153099313654381}